In [ ]:
!pip install -q transformers torch torchvision pillow matplotlib

In [ ]:
import torch
import matplotlib.pyplot as plt
from PIL import Image
from torchvision import transforms

In [ ]:
import requests
from io import BytesIO

url = "https://images.unsplash.com/photo-1552053831-71594a27632d"

response = requests.get(url)
image = Image.open(BytesIO(response.content)).convert("RGB")

display(image)

In [ ]:
image = image.resize((224, 224))

display(image)

In [ ]:
patch_size = 32

patches = []

for y in range(0, 224, patch_size):
    for x in range(0, 224, patch_size):
        patch = image.crop(
            (x, y, x + patch_size, y + patch_size)
        )
        patches.append(patch)

print("Number of patches:", len(patches))

In [ ]:
fig, axes = plt.subplots(7, 7, figsize=(10, 10))

for i, ax in enumerate(axes.flat):
    ax.imshow(patches[i])
    ax.set_title(f"P{i+1}")
    ax.axis("off")

plt.tight_layout()
plt.show()

In [ ]:
import torch
from transformers import CLIPProcessor, CLIPModel

In [ ]:
model_name = "openai/clip-vit-base-patch32"

clip_model = CLIPModel.from_pretrained(model_name)
processor = CLIPProcessor.from_pretrained(model_name)

In [ ]:
patch_inputs = processor(
    images=patches,
    return_tensors="pt"
)

In [ ]:
with torch.no_grad():
    outputs = clip_model.vision_model(**patch_inputs)

patch_embeddings = outputs.pooler_output

print("Patch embedding shape:", patch_embeddings.shape)

In [ ]:
!pip install -q transformers

In [ ]:
from transformers import BlipProcessor, BlipForConditionalGeneration

In [ ]:
caption_processor = BlipProcessor.from_pretrained(
    "Salesforce/blip-image-captioning-base"
)

caption_model = BlipForConditionalGeneration.from_pretrained(
    "Salesforce/blip-image-captioning-base"
)

In [ ]:
inputs = caption_processor(
    images=image,
    return_tensors="pt"
)

with torch.no_grad():
    output = caption_model.generate(**inputs, max_new_tokens=30)

caption = caption_processor.decode(
    output[0],
    skip_special_tokens=True
)

print("Generated Caption:")
print(caption)

In [ ]:
print("Patch embedding shape:", patch_embeddings.shape)

In [ ]:
with torch.no_grad():
    vision_outputs = clip_model.vision_model(**patch_inputs)

print("Vision output shape:", vision_outputs.last_hidden_state.shape)